<a href="https://colab.research.google.com/github/mdtanjimhasanashik/DeepGene-TB/blob/main/gene_expression_preprocessing_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind

# Load gene expression dataset
df = pd.read_csv("/content/drive/MyDrive/Data_Set/RawData_MetaData.csv")  # Update path as needed
print("Column names:", df.columns.tolist())
print("Dataset shape:", df.shape)

# Identify sample and label columns
sample_col = df.columns[0]
label_col = df.columns[-1]

# Extract expression data and labels
gene_expression = df.iloc[:, 1:-1]  # All genes (exclude sample and label)
labels = df[label_col]

# Split into condition groups
group1 = gene_expression[labels == labels.unique()[0]]
group2 = gene_expression[labels == labels.unique()[1]]

# --- Statistical Significance: p-value ---
p_values = [ttest_ind(group1[col], group2[col], equal_var=False).pvalue for col in gene_expression.columns]
pval_df = pd.DataFrame({"Gene": gene_expression.columns, "p_value": p_values})

# --- Biological Significance: log2 fold change ---
group1_mean = group1.mean()
group2_mean = group2.mean()
fold_change = group1_mean / group2_mean.replace(0, np.nan)
log2fc = np.log2(fold_change.replace(0, np.nan))
fc_df = pd.DataFrame({"Gene": gene_expression.columns, "log2fc": log2fc})

# --- Merge both results ---
deg_df = pd.merge(pval_df, fc_df, on="Gene")

# --- Combined Significance ---
deg_df["Combined_Significant"] = (deg_df["p_value"] < 0.05) & (deg_df["log2fc"].abs() > 1)

# --- Filter only combined significant DEGs ---
combined_sig_genes = deg_df[deg_df["Combined_Significant"]].copy()

# --- Upregulated (log2fc > 1) = 1, Downregulated (log2fc < -1) = 0 ---
combined_sig_genes["Target"] = np.where(combined_sig_genes["log2fc"] > 1, 1, 0)

# --- Print up/down counts ---
up_count = (combined_sig_genes["Target"] == 1).sum()
down_count = (combined_sig_genes["Target"] == 0).sum()

print("\n=== DEG CLASSIFICATION ===")
print(f"Upregulated DEGs (Target=1): {up_count}")
print(f"Downregulated DEGs (Target=0): {down_count}")

# --- Final dataframe with gene names, log2fc, p_value, target ---
final_deg_df = combined_sig_genes[["Gene", "log2fc", "p_value", "Target"]]
print("\nSample of final DEGs dataframe:")
print(final_deg_df.head())

# --- Save to CSV ---
final_deg_df.to_csv("DEGs_Combined_UpDown_With_Target.csv", index=False)

# Optional: if you want a gene expression matrix with target info:
selected_genes = final_deg_df["Gene"].tolist()
expr_only = df[selected_genes]
expr_with_target = expr_only.T.copy()
expr_with_target["log2fc"] = combined_sig_genes.set_index("Gene")["log2fc"]
expr_with_target["p_value"] = combined_sig_genes.set_index("Gene")["p_value"]
expr_with_target["Target"] = combined_sig_genes.set_index("Gene")["Target"]
expr_with_target.to_csv("DEGs_Expression_Log2FC_pvalue_Target.csv")

Column names: ['Sample_Names', 'TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112', 'FGR', 'CFH', 'FUCA2', 'GCLC', 'NFYA', 'STPG1', 'NIPAL3', 'LAS1L', 'ENPP4', 'SEMA3F', 'CFTR', 'ANKIB1', 'CYP51A1', 'KRIT1', 'RAD52', 'BAD', 'LAP3', 'CD99', 'HS3ST1', 'AOC1', 'WNT16', 'HECW1', 'MAD1L1', 'LASP1', 'SNX11', 'TMEM176A', 'M6PR', 'KLHL13', 'CYP26B1', 'ICA1', 'DBNDD1', 'ALS2', 'CASP10', 'CFLAR', 'TFPI', 'NDUFAF7', 'RBM5', 'MTMR7', 'SLC7A2', 'ARF5', 'SARM1', 'POLDIP2', 'PLXND1', 'AK2', 'CD38', 'FKBP4', 'KDM1A', 'RBM6', 'CAMKK1', 'RECQL', 'VPS50', 'HSPB6', 'ARHGAP33', 'NDUFAB1', 'PDK4', 'SLC22A16', 'ZMYND10', 'ABCB5', 'ARX', 'SLC25A13', 'ST7', 'CDC27', 'SLC4A1', 'CALCR', 'HCCS', 'DVL2', 'PRSS22', 'UPF1', 'SKAP2', 'SLC25A5', 'MCUB', 'HOXA11', 'POLR2J', 'DHX33', 'MEOX1', 'THSD7A', 'LIG3', 'RPAP3', 'ACSM3', 'LOC81691', 'CIAPIN1', 'SPPL2B', 'FAM214B', 'COPZ2', 'PRKAR2B', 'MSL3', 'CREBBP', 'TSPOAP1', 'MPO', 'PON1', 'GCFC2', 'WDR54', 'CROT', 'ABCB4', 'KMT2E', 'RHBDD2', 'SOX8', 'IBTK', 'ZNF195', 'MYCBP2', 'FB

In [ ]:
from statsmodels.stats.multitest import multipletests

# Assuming you have raw p-values in `combined_sig_genes["p_value"]`
_, adj_pvals, _, _ = multipletests(combined_sig_genes["p_value"], method='fdr_bh')
combined_sig_genes["adj_p_value"] = adj_pvals


In [ ]:
# --- Statistical Significance: p-value ---
p_values = [ttest_ind(group1[col], group2[col], equal_var=False).pvalue for col in gene_expression.columns]
pval_df = pd.DataFrame({"Gene": gene_expression.columns, "p_value": p_values})

In [ ]:
from statsmodels.stats.multitest import multipletests

# --- Statistical Significance: p-value + adjusted p-value ---
p_values = [ttest_ind(group1[col], group2[col], equal_var=False).pvalue for col in gene_expression.columns]
adjusted_pvals = multipletests(p_values, method='fdr_bh')[1]
pval_df = pd.DataFrame({
    "Gene": gene_expression.columns,
    "p_value": p_values,
    "adj_p_value": adjusted_pvals
})

In [ ]:
final_deg_df = combined_sig_genes[["Gene", "log2fc", "p_value", "adj_p_value", "Target"]]

In [ ]:
final_deg_df = combined_sig_genes[["Gene", "log2fc", "p_value", "adj_p_value", "Target"]]

In [ ]:
final_deg_df

,Gene,log2fc,p_value,adj_p_value,Target
5,FGR,2.312334,1.424200e-08,3.656048e-07,1
13,ENPP4,1.055971,3.509842e-02,4.016773e-02,1
65,ST7,-1.883971,1.298770e-05,1.371826e-04,0
116,OSBPL7,-1.201447,3.824219e-04,1.882407e-03,0
122,CACNG3,-1.255598,1.077081e-04,7.161705e-04,0
...,...,...,...,...,...
27960,MIR7515,2.000880,4.434787e-02,4.647932e-02,1
28015,LOC101929989,-1.143389,4.102652e-02,4.439796e-02,0
28023,ACR.1,-1.120220,1.215591e-02,2.073354e-02,0
28056,SYT14.1,-1.124801,5.214655e-03,1.205852e-02,0


In [ ]:
# Make sure log2FC column is absolute for sorting
combined_sig_genes["abs_log2FC"] = combined_sig_genes["log2fc"].abs()

# Sort by absolute log2 fold change
top_150_degs = combined_sig_genes.sort_values(by="abs_log2FC", ascending=False).head(150)

# Drop the helper column if you want
top_150_degs = top_150_degs.drop(columns=["abs_log2FC"])

# Save to CSV (optional)
top_150_degs.to_csv("Top_150_DEGs_by_log2FC.csv", index=False)

# Print summary
print("Top 150 DEGs selected based on absolute log2 fold change (|log2FC|):")
print(top_150_degs[["Gene", "log2fc", "p_value", "adj_p_value", "Target"]].head())

Top 150 DEGs selected based on absolute log2 fold change (|log2FC|):
            Gene    log2fc   p_value  adj_p_value  Target
27004  TBC1D3H.1 -9.047748  0.048561     0.049331       0
25918    TBC1D3H -9.047748  0.048561     0.049331       0
24109    TBC1D3G -9.043913  0.049162     0.049712       0
26388  TBC1D3G.1 -9.043913  0.049162     0.049712       0
27311   OR4N3P.2 -8.511147  0.045430     0.047199       0


In [ ]:
Top_150_DEGs=pd.read_csv("/content/Top_150_DEGs_by_log2FC.csv")

In [ ]:
Top_150_DEGs

,Gene,p_value,log2fc,Combined_Significant,Target,adj_p_value
0,TBC1D3H.1,0.048561,-9.047748,True,0,0.049331
1,TBC1D3H,0.048561,-9.047748,True,0,0.049331
2,TBC1D3G,0.049162,-9.043913,True,0,0.049712
3,TBC1D3G.1,0.049162,-9.043913,True,0,0.049712
4,OR4N3P.2,0.045430,-8.511147,True,0,0.047199
...,...,...,...,...,...,...
145,RSAD2,0.006387,-2.824846,True,0,0.013813
146,VWC2L-IT1,0.011494,-2.823282,True,0,0.020043
147,LTA.5,0.000003,-2.813293,True,0,0.000038
148,LTA.4,0.000003,-2.813236,True,0,0.000038


In [ ]:
final_150_deg = Top_150_DEGs[["Gene", "log2fc", "p_value", "adj_p_value", "Target"]]

In [ ]:
final_150_deg

,Gene,log2fc,p_value,adj_p_value,Target
0,TBC1D3H.1,-9.047748,0.048561,0.049331,0
1,TBC1D3H,-9.047748,0.048561,0.049331,0
2,TBC1D3G,-9.043913,0.049162,0.049712,0
3,TBC1D3G.1,-9.043913,0.049162,0.049712,0
4,OR4N3P.2,-8.511147,0.045430,0.047199,0
...,...,...,...,...,...
145,RSAD2,-2.824846,0.006387,0.013813,0
146,VWC2L-IT1,-2.823282,0.011494,0.020043,0
147,LTA.5,-2.813293,0.000003,0.000038,0
148,LTA.4,-2.813236,0.000003,0.000038,0


In [ ]:
final_150_deg ['Target'].value_counts()

,count
Target,
0,141
1,9


In [ ]:
final_150_deg.to_csv('/content/drive/My Drive/Data_Set/DENV_gene/final_150_degs.csv')

In [ ]:
Final_DEGs=pd.read_csv("/content/drive/MyDrive/Data_Set/DENV_gene/DEGs_Final.csv")

In [ ]:
Final_DEGs

,Unnamed: 0,Gene,0,1,2,3,4,5,6,7,...,2237,2238,2239,2240,2241,2242,2243,log2fc,p_value,target
0,0,FGR,0.0000,0.0000,0.0000,0.000,2.9110,16.9374,0.000,1.2777,...,4.0247,19.8922,18.1169,0.0000,0.0,0.0000,8.6683,2.312334,1.424200e-08,1.0
1,1,ENPP4,2.8671,11.4588,0.0000,0.000,13.4420,0.0000,0.000,617.6932,...,0.0000,0.8117,0.0000,60.5167,0.0,0.0000,0.0000,1.055971,3.509842e-02,1.0
2,2,ST7,9.6391,44.6741,48.9423,50.224,45.3135,0.0000,87.121,0.7783,...,6.2787,0.0000,9.3375,15.3363,0.0,4.3941,7.8650,-1.883971,1.298770e-05,0.0
3,3,OSBPL7,0.0000,0.3913,10.5258,0.000,6.4898,1.5395,0.000,0.0000,...,797.1408,279.1432,6.5387,0.6939,0.0,2.4243,2.8046,-1.201447,3.824219e-04,0.0
4,4,CACNG3,0.0000,0.8965,0.0000,0.000,2.4045,0.0000,0.000,0.0000,...,0.0000,0.0000,0.8075,0.0000,0.0,0.0000,0.0000,-1.255598,1.077081e-04,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,2023,MIR7515,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,2.000880,4.434787e-02,1.0
2024,2024,LOC101929989,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,32.7870,0.0,0.0000,0.0000,-1.143389,4.102652e-02,0.0
2025,2025,ACR.1,0.0000,0.0000,0.0000,0.228,0.0000,2.0852,0.000,4.7458,...,0.0000,0.0000,0.0000,0.0000,0.0,2.5423,0.3553,-1.120220,1.215591e-02,0.0
2026,2026,SYT14.1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,9.1070,0.0000,0.0,0.0000,0.0000,-1.124801,5.214655e-03,0.0


In [ ]:
DFM = pd.merge(Final_DEGs, final_deg_df[["Gene", "adj_p_value"]], on="Gene", how="left")

In [ ]:
DFM

,Unnamed: 0,Gene,0,1,2,3,4,5,6,7,...,2238,2239,2240,2241,2242,2243,log2fc,p_value,target,adj_p_value
0,0,FGR,0.0000,0.0000,0.0000,0.000,2.9110,16.9374,0.000,1.2777,...,19.8922,18.1169,0.0000,0.0,0.0000,8.6683,2.312334,1.424200e-08,1.0,3.656048e-07
1,1,ENPP4,2.8671,11.4588,0.0000,0.000,13.4420,0.0000,0.000,617.6932,...,0.8117,0.0000,60.5167,0.0,0.0000,0.0000,1.055971,3.509842e-02,1.0,4.016773e-02
2,2,ST7,9.6391,44.6741,48.9423,50.224,45.3135,0.0000,87.121,0.7783,...,0.0000,9.3375,15.3363,0.0,4.3941,7.8650,-1.883971,1.298770e-05,0.0,1.371826e-04
3,3,OSBPL7,0.0000,0.3913,10.5258,0.000,6.4898,1.5395,0.000,0.0000,...,279.1432,6.5387,0.6939,0.0,2.4243,2.8046,-1.201447,3.824219e-04,0.0,1.882407e-03
4,4,CACNG3,0.0000,0.8965,0.0000,0.000,2.4045,0.0000,0.000,0.0000,...,0.0000,0.8075,0.0000,0.0,0.0000,0.0000,-1.255598,1.077081e-04,0.0,7.161705e-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,2023,MIR7515,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,2.000880,4.434787e-02,1.0,4.647932e-02
2024,2024,LOC101929989,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,32.7870,0.0,0.0000,0.0000,-1.143389,4.102652e-02,0.0,4.439796e-02
2025,2025,ACR.1,0.0000,0.0000,0.0000,0.228,0.0000,2.0852,0.000,4.7458,...,0.0000,0.0000,0.0000,0.0,2.5423,0.3553,-1.120220,1.215591e-02,0.0,2.073354e-02
2026,2026,SYT14.1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,9.1070,0.0000,0.0,0.0000,0.0000,-1.124801,5.214655e-03,0.0,1.205852e-02


In [ ]:
# Move 'Target' column to the end
target_col = DFM .pop("target")  # Remove and store the 'Target' column
DFM["target"] = target_col      # Add it back at the end

# Save to CSV (optional)
DFM.to_csv("DEGs_set.csv", index=False)

# Print summary
print("DEGs_set (Target column moved to end):")
print(DFM .head())

DEGs_set (Target column moved to end):
   Unnamed: 0    Gene       0        1        2       3        4        5  \
0           0     FGR  0.0000   0.0000   0.0000   0.000   2.9110  16.9374   
1           1   ENPP4  2.8671  11.4588   0.0000   0.000  13.4420   0.0000   
2           2     ST7  9.6391  44.6741  48.9423  50.224  45.3135   0.0000   
3           3  OSBPL7  0.0000   0.3913  10.5258   0.000   6.4898   1.5395   
4           4  CACNG3  0.0000   0.8965   0.0000   0.000   2.4045   0.0000   

        6         7  ...      2238     2239     2240  2241    2242    2243  \
0   0.000    1.2777  ...   19.8922  18.1169   0.0000   0.0  0.0000  8.6683   
1   0.000  617.6932  ...    0.8117   0.0000  60.5167   0.0  0.0000  0.0000   
2  87.121    0.7783  ...    0.0000   9.3375  15.3363   0.0  4.3941  7.8650   
3   0.000    0.0000  ...  279.1432   6.5387   0.6939   0.0  2.4243  2.8046   
4   0.000    0.0000  ...    0.0000   0.8075   0.0000   0.0  0.0000  0.0000   

     log2fc       p_value   a

In [ ]:
DFM

,Unnamed: 0,Gene,0,1,2,3,4,5,6,7,...,2238,2239,2240,2241,2242,2243,log2fc,p_value,adj_p_value,target
0,0,FGR,0.0000,0.0000,0.0000,0.000,2.9110,16.9374,0.000,1.2777,...,19.8922,18.1169,0.0000,0.0,0.0000,8.6683,2.312334,1.424200e-08,3.656048e-07,1.0
1,1,ENPP4,2.8671,11.4588,0.0000,0.000,13.4420,0.0000,0.000,617.6932,...,0.8117,0.0000,60.5167,0.0,0.0000,0.0000,1.055971,3.509842e-02,4.016773e-02,1.0
2,2,ST7,9.6391,44.6741,48.9423,50.224,45.3135,0.0000,87.121,0.7783,...,0.0000,9.3375,15.3363,0.0,4.3941,7.8650,-1.883971,1.298770e-05,1.371826e-04,0.0
3,3,OSBPL7,0.0000,0.3913,10.5258,0.000,6.4898,1.5395,0.000,0.0000,...,279.1432,6.5387,0.6939,0.0,2.4243,2.8046,-1.201447,3.824219e-04,1.882407e-03,0.0
4,4,CACNG3,0.0000,0.8965,0.0000,0.000,2.4045,0.0000,0.000,0.0000,...,0.0000,0.8075,0.0000,0.0,0.0000,0.0000,-1.255598,1.077081e-04,7.161705e-04,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,2023,MIR7515,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,2.000880,4.434787e-02,4.647932e-02,1.0
2024,2024,LOC101929989,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,32.7870,0.0,0.0000,0.0000,-1.143389,4.102652e-02,4.439796e-02,0.0
2025,2025,ACR.1,0.0000,0.0000,0.0000,0.228,0.0000,2.0852,0.000,4.7458,...,0.0000,0.0000,0.0000,0.0,2.5423,0.3553,-1.120220,1.215591e-02,2.073354e-02,0.0
2026,2026,SYT14.1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,9.1070,0.0000,0.0,0.0000,0.0000,-1.124801,5.214655e-03,1.205852e-02,0.0


In [ ]:
DFM.to_csv('/content/drive/My Drive/Data_Set/DENV_gene/DEGs_set.csv')

In [ ]:
DFM

,Unnamed: 0,Gene,0,1,2,3,4,5,6,7,...,2238,2239,2240,2241,2242,2243,log2fc,p_value,adj_p_value,target
0,0,FGR,0.0000,0.0000,0.0000,0.000,2.9110,16.9374,0.000,1.2777,...,19.8922,18.1169,0.0000,0.0,0.0000,8.6683,2.312334,1.424200e-08,3.656048e-07,1.0
1,1,ENPP4,2.8671,11.4588,0.0000,0.000,13.4420,0.0000,0.000,617.6932,...,0.8117,0.0000,60.5167,0.0,0.0000,0.0000,1.055971,3.509842e-02,4.016773e-02,1.0
2,2,ST7,9.6391,44.6741,48.9423,50.224,45.3135,0.0000,87.121,0.7783,...,0.0000,9.3375,15.3363,0.0,4.3941,7.8650,-1.883971,1.298770e-05,1.371826e-04,0.0
3,3,OSBPL7,0.0000,0.3913,10.5258,0.000,6.4898,1.5395,0.000,0.0000,...,279.1432,6.5387,0.6939,0.0,2.4243,2.8046,-1.201447,3.824219e-04,1.882407e-03,0.0
4,4,CACNG3,0.0000,0.8965,0.0000,0.000,2.4045,0.0000,0.000,0.0000,...,0.0000,0.8075,0.0000,0.0,0.0000,0.0000,-1.255598,1.077081e-04,7.161705e-04,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,2023,MIR7515,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,0.0000,0.0,0.0000,0.0000,2.000880,4.434787e-02,4.647932e-02,1.0
2024,2024,LOC101929989,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,0.0000,32.7870,0.0,0.0000,0.0000,-1.143389,4.102652e-02,4.439796e-02,0.0
2025,2025,ACR.1,0.0000,0.0000,0.0000,0.228,0.0000,2.0852,0.000,4.7458,...,0.0000,0.0000,0.0000,0.0,2.5423,0.3553,-1.120220,1.215591e-02,2.073354e-02,0.0
2026,2026,SYT14.1,0.0000,0.0000,0.0000,0.000,0.0000,0.0000,0.000,0.0000,...,0.0000,9.1070,0.0000,0.0,0.0000,0.0000,-1.124801,5.214655e-03,1.205852e-02,0.0
